# Training DeepSeek: loss, AdamW, and learning-rate scheduling

This notebook runs one FP32 optimization smoke test with the combined main+MTP objective, AdamW parameter groups, warmup/cosine scheduling, clipping, and expert-bias balancing.

In [1]:
import sys
from pathlib import Path

project_root = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / 'src' / 'model').is_dir()),
    None,
)
if project_root is None:
    raise RuntimeError('Could not find the repository root')
sys.path.insert(0, str(project_root))

import torch

from src.model import DeepSeekConfig, DeepSeekWithMTP
from src.training import MoELoadBalancer, create_optimizer, train_step

torch.manual_seed(25)

In [2]:
config = DeepSeekConfig(
    vocab_size=256, max_seq_len=32, dim=64,
    n_layers=3, n_dense_layers=1, n_heads=4,
    q_lora_rank=16, kv_lora_rank=8,
    qk_nope_head_dim=8, qk_rope_head_dim=4, v_head_dim=8,
    inter_dim=128, moe_inter_dim=32,
    n_routed_experts=4, n_shared_experts=1,
    n_activated_experts=2,
)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = DeepSeekWithMTP(config, mtp_weight=0.3).to(device)
optimizer = create_optimizer(model)
tokens = torch.randint(0, config.vocab_size, (2, 17), device=device)
print('device:', device)
print('parameter count:', sum(p.numel() for p in model.parameters()))

device: cpu
parameter count: 179808


In [3]:
with MoELoadBalancer(model, gamma=0.01) as balancer:
    history = []
    for step in range(1, 6):
        metrics = train_step(
            model, optimizer, balancer, tokens,
            step=step, total_steps=5, warmup_steps=2,
        )
        history.append(metrics)
        print(
            f"step {step} | loss {metrics['loss']:.4f} | "
            f"main {metrics['main_loss']:.4f} | "
            f"mtp {metrics['mtp_loss']:.4f} | "
            f"lr {metrics['lr']:.7f} | "
            f"grad {metrics['grad_norm']:.3f}"
        )

step 1 | loss 7.3178 | main 5.6040 | mtp 5.7124 | lr 0.0001500 | grad 1.856
step 2 | loss 7.2754 | main 5.5747 | mtp 5.6691 | lr 0.0003000 | grad 1.858
step 3 | loss 7.1891 | main 5.5154 | mtp 5.5792 | lr 0.0002325 | grad 1.855
step 4 | loss 7.1148 | main 5.4620 | mtp 5.5093 | lr 0.0000975 | grad 1.852
step 5 | loss 7.0933 | main 5.4496 | mtp 5.4788 | lr 0.0000300 | grad 1.851


The first two steps warm up the learning rate. The final steps are in the cosine-decay phase. The balancing update runs after AdamW and includes every router registered inside the main model and MTP block.

In [4]:
assert all(torch.isfinite(torch.tensor(item['loss'])) for item in history)
assert history[0]['lr'] < history[1]['lr']
assert len(history[-1]['load_ratios']) == 3
print('optimization smoke checks passed')

optimization smoke checks passed
